# W3D5 — Tuning an MLP, and the Week's Reckoning — Guided

**Week 3 · Day 5 · Deep Learning & Neural Networks** · Lab

Two halves today, and they have different jobs.

The first is **diagnosis**. Four loss curves, four different faults, and the vocabulary for naming
them on sight: a learning rate too high, a learning rate too low, overfitting, and a model that has
learned nothing at all and says so with the number 0.6931. Then you reproduce the overfitting curve
on purpose, and try three standard fixes on it — dropout, early stopping, a different optimiser —
and record which of them actually worked on **this** data rather than which one is supposed to.

The second is **the reckoning**. On Monday you wrote down 0.7812 ± 0.0054 from a boosted tree. Today
your best network goes next to it on the same data, with both spreads, and you report what happened.
On twenty-three tabular columns the tree usually wins. If it does, that is the result — and saying so
plainly is the marked part of this lab.

You leave with `experiments.parquet` and `best_model.pt`. **The capstone brief is released today**,
and `experiments.parquet` is the shape of the experiment log it asks for.

**Time budget:** ~115 minutes. Sections 1–2 are the lab; Section 3 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٣ اليوم ٥ — ضبط الشبكة، وحساب الأسبوع

**الأسبوع ٣ · اليوم ٥ · التعلّم العميق والشبكات العصبية** · معمل عملي

نصفان اليوم، ولكلٍّ منهما عمل مختلف.

الأول **التشخيص**: أربعة منحنيات خسارة وأربعة أعطال مختلفة، والمفردات التي تسمّيها بها من النظرة
الأولى: معدّل تعلّم مرتفع، ومعدّل منخفض، وفرط مطابقة، ونموذج لم يتعلّم شيئًا ويقول ذلك بالرقم
٠٫٦٩٣١. ثم تُعيد إنتاج منحنى فرط المطابقة عن قصد، وتُجرّب عليه ثلاثة علاجات معيارية — Dropout
والإيقاف المبكر ومُحسِّنًا آخر — وتسجّل أيّها نفع على **هذه** البيانات لا أيّها يُفترض أن ينفع.

والثاني **الحساب**. فقد كتبت يوم الاثنين ٠٫٧٨١٢ ± ٠٫٠٠٥٤ من شجرة معزَّزة. واليوم توضع أفضل شبكة عندك
بجانبه على البيانات نفسها بمدى تفاوت كلٍّ منهما، وتُبلّغ بما حدث. وعلى ثلاثة وعشرين عمودًا جدوليًا
تفوز الشجرة عادةً. فإن فازت فتلك هي النتيجة — وقولها صراحةً هو الجزء الذي يُقيَّم في هذا المعمل.

ستخرج بملفَّي `experiments.parquet` و`best_model.pt`. **وتُنشر كرّاسة مشروع التخرّج اليوم**،
و`experiments.parquet` هو شكل سجلّ التجارب الذي تطلبه.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. القسمان الأول والثاني هما المعمل، والقسم الثالث إضافي يمكن إكماله في المنزل.

</div>

> **This is the guided version.** Most of the code is already here. Fill in the lines marked
> `# TODO`. If you want the full challenge, use the `_blank` version instead.

<div dir="rtl" align="right">

> **هذه النسخة الموجَّهة.** معظم الشيفرة موجودة، وعليك إكمال الأسطر المعلَّمة بـ `# TODO`.
> وإذا أردت التحدّي الكامل فاستخدم نسخة `_blank`.

</div>

## Learning objectives

By the end of this lab you can:

- Diagnose a training run from its loss curve, and name the fix: divide the learning rate by ten,
  raise it, stop earlier, or go and look at the data.
- Recognise 0.6931 on a balanced binary problem and know it means the model has learned nothing.
- Produce overfitting on purpose, and measure whether dropout fixed it on your data.
- Compare four optimisers under identical seeds, and prove they were identical by their initial loss.
- Implement early stopping that restores the **best** weights, and report the two numbers either side
  of it.
- Keep an experiment log where every row differs from another by exactly one hyperparameter.
- Save a model with its config, reload it, and reproduce its recorded score exactly.
- Put a neural network next to a boosted tree with both spreads, and report which won without
  flattering either.

<div dir="rtl" align="right">

## أهداف التعلّم

بنهاية هذا المعمل ستكون قادرًا على:

- تشخيص تدريب من منحنى خسارته وتسمية العلاج: اقسم معدّل التعلّم على عشرة، أو ارفعه، أو أوقِف مبكرًا،
  أو اذهب وانظر في البيانات.
- التعرّف على ٠٫٦٩٣١ في مسألة ثنائية متوازنة ومعرفة أنها تعني أن النموذج لم يتعلّم شيئًا.
- إنتاج فرط المطابقة عن قصد، وقياس هل عالجه Dropout على بياناتك.
- مقارنة أربعة مُحسِّنات ببذور متطابقة، وإثبات تطابقها بخسارتها الابتدائية.
- تنفيذ إيقاف مبكر يستعيد **أفضل** الأوزان، وعرض الرقمين على جانبيه.
- حفظ سجلّ تجارب يختلف فيه كل صف عن غيره بمعامل واحد بالضبط.
- حفظ نموذج مع إعداده وإعادة تحميله وإعادة إنتاج نتيجته المسجّلة بالضبط.
- وضع شبكة عصبية بجانب شجرة معزَّزة بمدى تفاوت كلٍّ منهما، وقول أيّهما فاز بلا مجاملة لأيٍّ منهما.

</div>


## About the data

**Two datasets today, for two different jobs.**

**`moons_toy`** — 500 rows × 3 columns, CC0. The same file you have used since Tuesday. It trains in
seconds, so it is where you run experiments: the overfitting demonstration, dropout, the four
optimisers, early stopping. Today you deliberately train on only **100** of its rows and validate on
the other 400 — starving the model of data is the most reliable way to produce overfitting, and
overfitting is the thing this lab has to have in front of it.

**`credit_default`** — 30,000 rows × 25 columns, UCI, CC BY 4.0. Monday's dataset, for the honest
comparison. 23 features, 22.1% positive, measured with ROC AUC on the same five folds Monday used.

You also load **`tree_baseline.json`** from Monday. If you did not finish that lab, `load_artefact`
falls back to the reference copy and prints a note.

**Watch out:** the two halves of this lab are not comparable with each other, and nothing here tries
to make them be. `moons_toy` is 2-D and balanced; `credit_default` is 23-D and imbalanced. A
hyperparameter that helps on the first has no obligation to help on the second, and one of the
lessons of the day is watching that happen.

<div dir="rtl" align="right">

## عن البيانات

**مجموعتان اليوم لعملين مختلفين.**

**`moons_toy`** — ٥٠٠ صف × ٣ أعمدة، رخصة CC0. الملف نفسه الذي تستخدمه منذ الثلاثاء. ويتدرّب في
ثوانٍ، فهو موضع التجارب: إظهار فرط المطابقة، وDropout، والمُحسِّنات الأربعة، والإيقاف المبكر. واليوم
تتدرّب عن قصد على **مئة** صف منه وتتحقّق على الأربعمئة الباقية — فتجويع النموذج من البيانات أوثق طريقة
لإنتاج فرط المطابقة، وفرط المطابقة هو ما يجب أن يكون أمام هذا المعمل.

**`credit_default`** — ٣٠٬٠٠٠ صف × ٢٥ عمودًا، من UCI، رخصة CC BY 4.0. بيانات الاثنين للمقارنة
الصادقة: ٢٣ خاصية، و٢٢٫١٪ موجبة، مقيسة بالمساحة تحت منحنى ROC على الطيّات الخمس نفسها التي استخدمها
الاثنين.

وتُحمّل أيضًا **`tree_baseline.json`** من الاثنين. فإن لم تُكمل ذلك المعمل رجعت `load_artefact` إلى
النسخة المرجعية وطبعت ملاحظة.

**انتبه:** نصفا هذا المعمل غير قابلين للمقارنة أحدهما بالآخر، ولا شيء هنا يحاول جعلهما كذلك. فـ
`moons_toy` ثنائية البعد ومتوازنة، و`credit_default` ذات ٢٣ بعدًا وغير متوازنة. والمعامل الذي ينفع في
الأولى غير ملزَم بأن ينفع في الثانية، ومن دروس اليوم أن ترى ذلك يحدث.

</div>


## Setup

Run the cell below first. It loads both datasets, Monday's baseline, and fixes the split and the
seed every experiment in this notebook shares.

<div dir="rtl" align="right">

## الإعداد

شغّل الخليّة أدناه أولًا. تُحمّل المجموعتين وخط أساس الاثنين، وتُثبّت التقسيم والبذرة اللذين تشترك
فيهما كل تجربة في هذا الدفتر.

</div>


In [ ]:
# === AIEP portable setup — works locally (Miniconda + uv) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, report

ensure("torch", "scikit-learn", "matplotlib", "pyarrow")
seed_everything(42)

import copy
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

from aiep.viz import use_course_style, PALETTE
use_course_style()

SEED = 42

# --- moons_toy: 100 rows to train on, 400 to be judged by --------------------------
moons = pd.read_parquet(get_dataset("moons_toy"))
X_all = torch.tensor(moons[["x1", "x2"]].to_numpy(), dtype=torch.float32)
y_all = torch.tensor(moons["label"].to_numpy().reshape(-1, 1), dtype=torch.float32)

order = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = order[:100], order[100:]
X_train, y_train = X_all[train_idx], y_all[train_idx]
X_val, y_val = X_all[val_idx], y_all[val_idx]

# --- credit_default and Monday's number --------------------------------------------
credit = pd.read_parquet(get_dataset("credit_default"))
CREDIT_FEATURES = [c for c in credit.columns if c not in ("ID", "default_next_month")]
X_credit = credit[CREDIT_FEATURES].to_numpy(dtype=float)
y_credit = credit["default_next_month"].to_numpy(dtype=float)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tree_baseline = json.loads(load_artefact("tree_baseline.json").read_text(encoding="utf-8"))

print(f"moons_toy: {len(X_train)} training rows, {len(X_val)} validation rows")
print(f"credit_default: {len(credit):,} rows x {len(CREDIT_FEATURES)} features")
print(f"\nMonday's baseline: {tree_baseline['best_model']}")
print(f"  AUC {tree_baseline['cv_mean']:.4f} +/- {tree_baseline['cv_std']:.4f}")
print("\n", versions())

## Section 1 — Warm-up: diagnose four runs  (≈25 min)

Everything in this section already works.

The cell below reproduces the four curves from this morning — A, B, C and D — as real training runs
with a pinned seed, not drawings. Each one changed exactly one thing.

Plot them, then write your diagnosis for each in the markdown cell that follows, **before** you run
the answer key in the cell after that. Writing it down first is the exercise; reading the answer with
your own guess already on the page is what makes it stick.

<div dir="rtl" align="right">

## القسم الأول — التهيئة: شخِّص أربعة تشغيلات (نحو ٢٥ دقيقة)

كل ما في هذا القسم يعمل أصلًا.

تُعيد الخليّة أدناه إنتاج منحنيات الصباح الأربعة — أ وب وج ود — كتشغيلات تدريب حقيقية ببذرة ثابتة لا
كرسوم. وكل واحد منها غيّر شيئًا واحدًا بالضبط.

ارسمها، ثم اكتب تشخيصك لكل واحد في خليّة النصّ التالية **قبل** أن تشغّل مفتاح الإجابات في الخليّة
بعدها. فالكتابة أولًا هي التمرين، وقراءة الجواب وتخمينك مكتوب على الورقة هي ما يُثبّته.

</div>


In [ ]:
def synthetic(n=1400, d=40, seed=0):
    """The same generated data the morning's curves came from."""
    g = torch.Generator().manual_seed(seed)
    Xs = torch.randn(n, d, generator=g)
    w = torch.randn(d, 1, generator=g)
    return Xs, ((Xs @ w + 1.2 * torch.randn(n, 1, generator=g)) > 0).float()


def full_batch_run(lr, epochs, ntrain=120, seed=0):
    """One gradient step per epoch, so a bad learning rate shows up immediately."""
    Xs, ys = synthetic(600, 20, seed)
    Xtr, ytr, Xva, yva = Xs[:ntrain], ys[:ntrain], Xs[400:], ys[400:]
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1))
    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    train, val = [], []
    for _ in range(epochs):
        model.train()
        optimiser.zero_grad()
        criterion(model(Xtr), ytr).backward()
        optimiser.step()
        model.eval()
        with torch.no_grad():
            train.append(criterion(model(Xtr), ytr).item())
            val.append(criterion(model(Xva), yva).item())
    return train, val


def mini_batch_run(epochs=80, ntrain=200, seed=0, lr=1e-3):
    """A wide network on very little data, trained a long time."""
    Xs, ys = synthetic()
    Xtr, ytr, Xva, yva = Xs[:ntrain], ys[:ntrain], Xs[1000:], ys[1000:]
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    torch.manual_seed(seed)
    model = nn.Sequential(nn.Linear(40, 256), nn.ReLU(),
                          nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 1))
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    train, val = [], []
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            optimiser.zero_grad()
            criterion(model(xb), yb).backward()
            optimiser.step()
        model.eval()
        with torch.no_grad():
            train.append(criterion(model(Xtr), ytr).item())
            val.append(criterion(model(Xva), yva).item())
    return train, val


CURVES = {
    "A": full_batch_run(lr=50.0, epochs=12),
    "B": full_batch_run(lr=1e-5, epochs=200),
    "C": mini_batch_run(),
    "D": ([float(np.log(2))] * 60, [float(np.log(2))] * 60),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (name, (train, val)) in zip(axes.ravel(), CURVES.items()):
    ax.plot(train, color=PALETTE[0], linewidth=2, label="training")
    ax.plot(val, color=PALETTE[1], linewidth=2, label="validation")
    if name == "A":
        ax.set_yscale("log")
    ax.set_title(f"curve {name}:  {train[0]:.4f} -> {train[-1]:.4f} "
                 f"over {len(train)} epochs")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
plt.tight_layout()
plt.show()

for name, (train, val) in CURVES.items():
    print(f"curve {name}: train {train[0]:.4f} -> {train[-1]:.4f} | "
          f"validation {val[0]:.4f} -> {val[-1]:.4f} | best validation {min(val):.4f} "
          f"at epoch {int(np.argmin(val))}")

**Your four diagnoses**, one line each — what is wrong, and what would you change?

- **A:**
- **B:**
- **C:**
- **D:**

<div dir="rtl" align="right">

**تشخيصاتك الأربعة**، سطر لكل واحد — ما العطل، وما الذي تغيّره؟

- **أ:**
- **ب:**
- **ج:**
- **د:**

</div>


In [ ]:
ANSWER_KEY = {
    "A": ("the learning rate is far too high. Each step overshoots and lands further up the "
          "other side, so the next gradient is larger and the next step is worse. "
          "Fix: divide the learning rate by 10, and again if it still diverges."),
    "B": ("the learning rate is far too low — 200 epochs moved the loss by 0.0003. "
          "Fix: raise it 10x. But check the other cause first: if every ReLU output is "
          "zero, the units are dead and no learning rate will help."),
    "C": ("overfitting. Training loss goes to ~0 while validation bottoms out early and "
          "then climbs. Fix: stop at the best epoch, get more data, or regularise — and "
          "measure which of those actually helped rather than assuming."),
    "D": ("nothing at all. 0.6931 is ln 2, which is exactly the loss of a model that "
          "outputs 0.5 for every input on a balanced problem. It has not learned a weak "
          "rule; it has learned no rule. Go and look at the data, the labels and the "
          "final layer before you touch a hyperparameter."),
}

for name, answer in ANSWER_KEY.items():
    print(f"curve {name}: {answer}\n")

print(f"and the number to memorise: ln 2 = {np.log(2):.4f}")
print(f"for a balanced k-class problem it is ln k — ln 5 = {np.log(5):.4f}")

## Section 2 — Core: six tasks  (≈60 min)

1. Reproduce curve C on purpose, on `moons_toy`.
2. Add dropout, and measure whether it helped. Report what you measure, not what you expect.
3. Four optimisers, identical seeds, one figure and a table.
4. Early stopping that restores the best weights.
5. `experiments.parquet` — eight rows, changing one thing at a time.
6. **The reckoning:** your best network against Monday's boosted tree.

<div dir="rtl" align="right">

## القسم الثاني — الأساسي: ست مهام (نحو ٦٠ دقيقة)

١. أعِد إنتاج المنحنى ج عن قصد على `moons_toy`.
٢. أضِف Dropout وقِس هل نفع. واعرض ما تقيسه لا ما تتوقّعه.
٣. أربعة مُحسِّنات ببذور متطابقة، وشكل واحد وجدول.
٤. إيقاف مبكر يستعيد أفضل الأوزان.
٥. `experiments.parquet` — ثمانية صفوف بتغيير شيء واحد في كل مرة.
٦. **الحساب:** أفضل شبكة عندك مقابل شجرة الاثنين المعزَّزة.

</div>


### Task 2.1 — build the overfitting machine

Take a network far larger than the problem needs — 256 hidden units, twice — give it 100 training
rows, and train it for 300 epochs. It has enough capacity to memorise those rows several times over,
and it will.

Watch for the shape from curve C: the training loss walks down towards zero, the validation loss
bottoms out somewhere early and then turns and climbs. **The turn is the moment the model stopped
learning the problem and started learning the rows.**

Write the training function once, with every knob as an argument. Every task after this one calls it
with one argument changed, which is what makes task 2.5's experiment log honest by construction
rather than by discipline.

<div dir="rtl" align="right">

### المهمة ٢٫١ — ابنِ آلة فرط المطابقة

خُذ شبكة أكبر بكثير مما تحتاجه المسألة — ٢٥٦ وحدة مخفية مرتين — وأعطِها مئة صف تدريب ودرّبها ثلاثمئة
حقبة. فسعتها تكفي لحفظ تلك الصفوف عدة مرات، وستحفظها.

وراقب الشكل من المنحنى ج: تمشي خسارة التدريب نازلة نحو الصفر، وتبلغ خسارة التحقّق قاعها مبكرًا ثم
تستدير وتصعد. **والاستدارة هي اللحظة التي توقّف فيها النموذج عن تعلّم المسألة وبدأ يحفظ الصفوف.**

اكتب دالة التدريب مرة واحدة وكل مقبض فيها وسيط. وكل مهمة بعدها تناديها بوسيط واحد مُغيَّر، وهذا ما
يجعل سجلّ تجارب المهمة ٢٫٥ صادقًا بحكم البناء لا بحكم الانضباط.

</div>


In [ ]:

# TODO: Write build_model(width, dropout, batch_norm, seed) -> nn.Sequential.
# مهمة: اكتب `build_model(width, dropout, batch_norm, seed)` ← `nn.Sequential`.


# TODO: Write train(...) returning a per-epoch DataFrame, and the fitted model.
# مهمة: اكتب `train(...)` تُرجع `DataFrame` لكل حقبة، والنموذج المُدرَّب.

# TODO: Run it: width 256, no dropout, 300 epochs. This is curve C, on your own data.
# مهمة: شغّلها: عرض ٢٥٦ وبلا Dropout و٣٠٠ حقبة. وهذا هو المنحنى ج على بياناتك أنت.

best_epoch = int(overfit_history["val_loss"].idxmin())
print(f"train loss:      {overfit_history.iloc[0]['train_loss']:.4f} -> "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}")
print(f"validation loss: {overfit_history.iloc[0]['val_loss']:.4f} -> "
      f"{overfit_history.iloc[-1]['val_loss']:.4f}")
print(f"best validation: {overfit_history['val_loss'].min():.4f} at epoch "
      f"{best_epoch}, and then it climbed for {300 - best_epoch} more epochs")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))

# TODO: Plot both curves and mark the best validation epoch.
# مهمة: ارسم المنحنيين وأشِر إلى حقبة أفضل تحقّق.

ax.set_xlabel("epoch")
ax.set_ylabel("binary cross-entropy")
ax.set_title("Curve C, reproduced on purpose")
ax.legend()
plt.show()

### Task 2.2 — add dropout, and measure it honestly

`nn.Dropout(0.3)` zeroes 30% of a layer's activations on every forward pass, chosen fresh each time.
No path through the network can be relied upon, so no unit is allowed to become indispensable.

Retrain the same network with `dropout=0.3` — one argument changed, everything else identical,
including the seed.

**Two numbers, and they do not move together.** The training loss should get **worse**: you have made
the network's job harder on purpose, and it is also being measured with units randomly silenced.
Whether the validation loss gets **better** is the actual experiment, and it is not guaranteed.

The lecture was explicit about this and so is the slide's own table: on a small tabular problem
dropout is a real tool and not a reliable win. Record both numbers and write down which way it went
on your data. If it made things worse, say so — that is a result, and a lab that only reports the
runs that worked is not a lab.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — أضِف Dropout وقِسه بصدق

يُصفّر `nn.Dropout(0.3)` ثلاثين بالمئة من تنشيطات الطبقة في كل مرور أمامي، ويُختارون من جديد كل مرة.
فلا يمكن الاعتماد على أي مسار في الشبكة، فلا يُسمح لأي وحدة أن تصير لا غنى عنها.

أعِد تدريب الشبكة نفسها بـ `dropout=0.3` — وسيط واحد مُغيَّر وكل ما عداه متطابق، والبذرة كذلك.

**ورقمان لا يتحرّكان معًا.** يُتوقّع أن **تسوء** خسارة التدريب: فقد صعّبت عمل الشبكة عن قصد، وهي
تُقاس أيضًا ووحداتها تُسكَت عشوائيًا. أما هل **تتحسّن** خسارة التحقّق فهي التجربة الفعلية، وليست
مضمونة.

وقد كانت المحاضرة صريحة في هذا وكذلك جدول الشريحة نفسه: ففي مسألة جدولية صغيرة يكون Dropout أداةً
حقيقية لا فوزًا موثوقًا. فسجّل الرقمين واكتب في أي اتجاه ذهب على بياناتك. وإن أساء فقُل ذلك — فهذه
نتيجة، والمعمل الذي لا يعرض إلا التشغيلات الناجحة ليس معملًا.

</div>


In [ ]:

# TODO: Retrain with dropout=0.3, changing nothing else.
# مهمة: أعِد التدريب بـ `dropout=0.3` بلا تغيير أي شيء آخر.

comparison = pd.DataFrame([
    {"run": "no dropout",
     "final_train": overfit_history.iloc[-1]["train_loss"],
     "final_val": overfit_history.iloc[-1]["val_loss"],
     "best_val": overfit_history["val_loss"].min(),
     "best_epoch": int(overfit_history["val_loss"].idxmin())},
    {"run": "dropout 0.3",
     "final_train": dropout_history.iloc[-1]["train_loss"],
     "final_val": dropout_history.iloc[-1]["val_loss"],
     "best_val": dropout_history["val_loss"].min(),
     "best_epoch": int(dropout_history["val_loss"].idxmin())},
])
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

train_worse = (comparison.iloc[1]["final_train"] > comparison.iloc[0]["final_train"])
val_better = (comparison.iloc[1]["best_val"] < comparison.iloc[0]["best_val"])
print(f"\ndropout made the training loss worse: {train_worse}   "
      f"<- this is the regularisation working")
print(f"dropout made the best validation loss better: {val_better}   "
      f"<- this is the question")

On this data, dropout made the training loss worse **and** the validation loss worse. Both numbers
went the same way, and only one of them was supposed to.

That is a real result and it is worth understanding rather than explaining away. `moons_toy` has
**two** input features. Dropout works by removing redundancy — forcing several paths to carry the
same information, so that no single one is indispensable. In a network fed by two columns there is
very little redundancy to spare, and silencing 30% of the units mostly destroys signal rather than
reducing reliance.

Dropout earns its reputation on wide, redundant inputs: pixels, embeddings, hundreds of columns.
The morning's curves came from 40 features, and there it moved the best validation loss from 0.2762
to 0.2673 — a small win, and the slide said plainly that the *final* loss got worse.

**So the honest report of this task is: dropout did not help here, and task 2.4 shows what did.**

<div dir="rtl" align="right">

على هذه البيانات، أساء Dropout خسارة التدريب **و**خسارة التحقّق معًا. فذهب الرقمان في الاتجاه نفسه،
وواحد منهما فقط كان يُفترض به ذلك.

وهذه نتيجة حقيقية تستحقّ الفهم لا التبرير. فـ `moons_toy` فيها **خاصيتان** مُدخلتان. وDropout يعمل
بإزالة الفائض — بإجبار عدة مسارات على حمل المعلومة نفسها فلا يكون أحدها لا غنى عنه. وفي شبكة تتغذّى
من عمودين لا فائض يُذكر، فإسكات ٣٠٪ من الوحدات يُتلف الإشارة أكثر مما يُقلّل الاعتماد.

ويكتسب Dropout سمعته على المدخلات العريضة الفائضة: البكسلات والتضمينات ومئات الأعمدة. ومنحنيات
الصباح جاءت من ٤٠ خاصية، وهناك حرّك أفضل خسارة تحقّق من ٠٫٢٧٦٢ إلى ٠٫٢٦٧٣ — فوز صغير، وقالت الشريحة
صراحةً إن الخسارة **النهائية** ساءت.

**فالتقرير الصادق لهذه المهمة: لم ينفع Dropout هنا، وتُريك المهمة ٢٫٤ ما نفع.**

</div>


### Task 2.3 — four optimisers, one seed

SGD, SGD with momentum, RMSprop, Adam. One line changes between them; everything else — the
architecture, the initial weights, the batch order, the number of epochs — is identical.

**Prove that it is identical**, do not assert it. If all four runs start from the same initial loss
to six decimal places, they started from the same weights on the same data. If they do not, your
comparison is measuring initialisation and you would not know.

Then the table: epochs to reach a validation loss of 0.2, and the best validation loss inside the
budget. Plain SGD may never arrive. That is the point of including it.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — أربعة مُحسِّنات وبذرة واحدة

SGD، وSGD مع الزخم، وRMSprop، وAdam. يتغيّر سطر واحد بينها، وكل ما عداه — المعمارية والأوزان
الابتدائية وترتيب الدفعات وعدد الحقب — متطابق.

**وأثبِت أنه متطابق** ولا تدّعِه. فإن بدأت التشغيلات الأربعة من الخسارة الابتدائية نفسها إلى ست خانات
عشرية فقد بدأت من الأوزان نفسها على البيانات نفسها. وإن لم تبدأ فمقارنتك تقيس التهيئة وأنت لا تدري.

ثم الجدول: عدد الحقب للوصول إلى خسارة تحقّق ٠٫٢، وأفضل خسارة تحقّق داخل الميزانية. وقد لا يصل SGD
العادي أبدًا، وهذا هو سبب إدراجه.

</div>


In [ ]:
OPTIMISERS = ["sgd", "momentum", "rmsprop", "adam"]
TARGET_LOSS = 0.2

# TODO: Train once per optimiser, 100 epochs, everything else identical.
# مهمة: درّب مرة لكل مُحسِّن، مئة حقبة، وكل ما عداه متطابق.

# TODO: Build the table: epochs to reach TARGET_LOSS, and the best validation loss.
# مهمة: ابنِ الجدول: الحقب اللازمة لبلوغ `TARGET_LOSS`، وأفضل خسارة تحقّق.

fig, ax = plt.subplots(figsize=(9, 4.5))
for name, colour in zip(OPTIMISERS, PALETTE):
    ax.plot(optimiser_runs[name]["val_loss"], color=colour, linewidth=2, label=name)
ax.axhline(TARGET_LOSS, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("epoch")
ax.set_ylabel("validation loss")
ax.set_title("One line changed, four different trainings")
ax.legend()
plt.show()

print(optimiser_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
same_start = optimiser_table["loss_before_training"].round(6).nunique() == 1
print(f"\nall four started from the same weights: {same_start}   "
      f"(training loss before the first step, to 6 decimals)")

### Task 2.4 — early stopping, and the number it is worth

Task 2.1 showed the model you wanted appearing early and then being trained past. Early stopping is
the fix, and it has two halves — only one of which people remember.

**Stop when validation stops improving.** `patience=10` means: after 10 epochs with no improvement,
give up.

**Restore the best weights, not the last ones.** These are different models. `model.load_state_dict(best_state)`
is the line people forget, and forgetting it turns early stopping into "training for slightly fewer
epochs", which is worth almost nothing.

Run with `patience=10`, then score the model twice: as it was at the final epoch, and with the best
weights restored. Report both.

<div dir="rtl" align="right">

### المهمة ٢٫٤ — الإيقاف المبكر والرقم الذي يساويه

أرَتك المهمة ٢٫١ النموذج الذي أردته يظهر مبكرًا ثم يُتجاوَز بالتدريب. والإيقاف المبكر هو العلاج، وله
نصفان لا يتذكّر الناس إلا واحدًا منهما.

**توقّف حين تتوقّف نتيجة التحقّق عن التحسّن.** فـ `patience=10` تعني: بعد عشر حقب بلا تحسّن، استسلم.

**واستعِد أفضل الأوزان لا آخرها.** فهذان نموذجان مختلفان. و`model.load_state_dict(best_state)` هو
السطر الذي ينساه الناس، ونسيانه يحوّل الإيقاف المبكر إلى «تدريب بحقب أقل قليلًا»، وهذا لا يساوي شيئًا
تقريبًا.

شغّل بـ `patience=10`، ثم قيّم النموذج مرتين: كما كان في الحقبة الأخيرة، وبأفضل الأوزان مستعادةً.
واعرض الرقمين.

</div>


In [ ]:
criterion = nn.BCEWithLogitsLoss()


def score(model, X, y):
    """Validation loss and accuracy, in eval mode, without building a graph."""
    model.eval()
    with torch.no_grad():
        logits = model(X)
        return (criterion(logits, y).item(),
                float(((logits > 0) == (y > 0.5)).float().mean()))


# TODO: Train with patience=10, then score the final weights and the restored best ones.
# مهمة: درّب بـ `patience=10`، ثم قيّم الأوزان النهائية ثم أفضل الأوزان المستعادة.

print(f"stopped after {len(stopped_history)} epochs of a 300-epoch budget")
print(f"  last epoch's weights:  val loss {final_loss:.4f}, "
      f"accuracy {final_accuracy:.3f}")
print(f"  best epoch's weights:  val loss {restored_loss:.4f}, "
      f"accuracy {restored_accuracy:.3f}")
print(f"\nrestoring the best weights was worth "
      f"{final_loss - restored_loss:.4f} of validation loss")
print(f"and against the 300-epoch run from task 2.1 "
      f"({overfit_history.iloc[-1]['val_loss']:.4f}), it is worth "
      f"{overfit_history.iloc[-1]['val_loss'] - restored_loss:.4f}")

Compare the last line with task 2.2's dropout result and the ranking is unambiguous.

Dropout, on this data, cost you validation loss. Early stopping, which added no hyperparameter to the
model at all and simply declined to keep training, recovered most of what 300 epochs of overfitting
had thrown away. **The best model already happened; training continued past it.**

This is the general shape of the lesson, and it survives outside this dataset: before reaching for a
regulariser, check whether you are simply keeping the wrong epoch. It is free, it always applies, and
it is the first thing to try.

<div dir="rtl" align="right">

قارن السطر الأخير بنتيجة Dropout في المهمة ٢٫٢ فيصير الترتيب لا لبس فيه.

فقد كلّفك Dropout على هذه البيانات خسارة تحقّق. أما الإيقاف المبكر — الذي لم يُضِف إلى النموذج أي
معامل، بل امتنع عن مواصلة التدريب فحسب — فاستعاد أكثر ما رمته ثلاثمئة حقبة من فرط المطابقة. **فالنموذج
الأفضل قد حدث أصلًا، والتدريب واصل بعده.**

وهذا هو شكل الدرس العام، ويصمد خارج هذه البيانات: قبل أن تمدّ يدك إلى مُنظِّم، تحقّق هل أنت ببساطة
تحتفظ بالحقبة الخطأ. فهذا مجّاني، وينطبق دائمًا، وهو أول ما تُجرّبه.

</div>


### Task 2.5 — the experiment log

Every number you have produced today is currently a print statement that will be gone the moment you
restart the kernel. Now write them down properly.

`experiments.parquet`: one row per run, with **every** hyperparameter, the seed, both losses and the
validation accuracy. At least eight rows, and — this is the rule that matters — **each row differs
from the baseline by exactly one hyperparameter.**

That rule is the transferable skill of the day and it is the one people abandon under time pressure.
Change two things, get a better number, and you have learned nothing: you cannot say which change
helped, and you cannot undo the one that hurt. The check at the end of this notebook walks every row against
the baseline and fails if any of them moved two knobs, because the instruction alone has never been
enough.

This runs eight trainings and takes about a minute.

<div dir="rtl" align="right">

### المهمة ٢٫٥ — سجلّ التجارب

كل رقم أنتجته اليوم هو الآن جملة طباعة تختفي لحظة إعادة تشغيل النواة. فاكتبها الآن كما ينبغي.

`experiments.parquet`: صف لكل تشغيل فيه **كل** معامل، والبذرة، والخسارتان، ودقة التحقّق. وثمانية صفوف
على الأقل، والقاعدة المهمّة: **يختلف كل صف عن خط الأساس بمعامل واحد بالضبط.**

وهذه القاعدة هي المهارة المنقولة في اليوم، وهي التي يتركها الناس تحت ضغط الوقت. فإن غيّرت شيئين وحصلت
على رقم أفضل فلم تتعلّم شيئًا: إذ لا تستطيع قول أي تغيير نفع، ولا تستطيع التراجع عن الذي ضرّ. ويفرض
الفحص في آخر هذا الدفتر ذلك، لأن التعليمة وحدها لم تكفِ يومًا.

وهذا يشغّل ثمانية تدريبات ويستغرق نحو دقيقة.

</div>


In [ ]:
BASE = {"width": 256, "dropout": 0.0, "batch_norm": False, "optimiser": "adam",
        "lr": 1e-3, "batch_size": 32, "epochs": 100, "seed": SEED}

VARIANTS = [("baseline", None, None),
            ("narrower", "width", 32),
            ("dropout 0.3", "dropout", 0.3),
            ("sgd", "optimiser", "sgd"),
            ("momentum", "optimiser", "momentum"),
            ("rmsprop", "optimiser", "rmsprop"),
            ("lr 1e-2", "lr", 1e-2),
            ("lr 1e-4", "lr", 1e-4),
            ("batch 128", "batch_size", 128)]

# TODO: Run every variant and collect one row per run, config included.
# مهمة: شغّل كل متغيّر واجمع صفًا لكل تشغيل مع إعداده.

columns = ["run", "changed", "value", "final_train_loss", "best_val_loss",
           "best_epoch", "final_val_accuracy"]
print(experiments[columns].to_string(index=False,
                                     float_format=lambda v: f"{v:.4f}"))

best_run = experiments.loc[experiments["best_val_loss"].idxmin()]
print(f"\nbest configuration: {best_run['run']} "
      f"(best validation loss {best_run['best_val_loss']:.4f})")
print(f"and it is the best of {len(experiments)} — say that when you report it")

### Task 2.6 — the reckoning

Everything until now was practice on 500 points in a plane. This is the question the week was built
around.

Train your best configuration on `credit_default` — 23 real columns, 30,000 rows, the same five folds
Monday used, scored with the same metric — and put the number next to `tree_baseline.json`.

Three things this task requires, and they are all about fairness rather than about networks:

- **Scale the features, inside the fold.** A network cares about scale; `LIMIT_BAL` runs to hundreds
  of thousands and `AGE` to about 70. Fit the `StandardScaler` on each fold's training rows only —
  W2D5's rule, and it still applies when the model changed.
- **Five folds, not one split.** Monday reported a mean and a spread. A single number compared against
  a mean ± std is not a comparison.
- **Report it either way.** If the tree wins, say the tree won.

This trains five networks and takes about half a minute.

<div dir="rtl" align="right">

### المهمة ٢٫٦ — الحساب

كل ما سبق تمرين على خمسمئة نقطة في مستوى. وهذا هو السؤال الذي بُني الأسبوع حوله.

درّب أفضل إعداد عندك على `credit_default` — ٢٣ عمودًا حقيقيًا و٣٠٬٠٠٠ صف والطيّات الخمس نفسها التي
استخدمها الاثنين والمقياس نفسه — وضَع الرقم بجانب `tree_baseline.json`.

وثلاثة أمور تتطلّبها هذه المهمة، وكلها عن الإنصاف لا عن الشبكات:

- **قيِّس الخصائص داخل الطيّة.** فالشبكة يهمّها المقياس؛ إذ يبلغ `LIMIT_BAL` مئات الآلاف و`AGE` نحو
  السبعين. ودرّب `StandardScaler` على صفوف تدريب كل طيّة وحدها — وهي قاعدة الأسبوع الثاني اليوم
  الخامس، وتبقى سارية وإن تغيّر النموذج.
- **خمس طيّات لا تقسيمًا واحدًا.** فقد عرض الاثنين متوسطًا ومدى تفاوت. والرقم الواحد مقارنًا بمتوسط
  ± انحراف ليس مقارنة.
- **وأبلِغ في الحالتين.** فإن فازت الشجرة فقُل إن الشجرة فازت.

وهذا يدرّب خمس شبكات ويستغرق نحو نصف دقيقة.

</div>


In [ ]:
CREDIT_CONFIG = {"width": 64, "dropout": 0.3, "optimiser": "adam", "lr": 1e-3,
                 "batch_size": 256, "epochs": 15, "seed": SEED}

# TODO: Cross-validate the network on credit_default, scaling inside each fold.
# مهمة: طبّق التحقّق المتقاطع للشبكة على `credit_default` مع التقييس داخل كل طيّة.

network_mean, network_std = float(fold_aucs.mean()), float(fold_aucs.std())
tree_mean, tree_std = tree_baseline["cv_mean"], tree_baseline["cv_std"]

print(f"\n{'model':<34}{'AUC':>10}{'spread':>10}")
print("-" * 54)
print(f"{'neural network (this notebook)':<34}{network_mean:>10.4f}"
      f"{network_std:>10.4f}")
print(f"{tree_baseline['best_model']:<34}{tree_mean:>10.4f}{tree_std:>10.4f}")
print(f"\ndifference: {network_mean - tree_mean:+.4f} of AUC in the network's favour")
overlap = abs(network_mean - tree_mean) < (network_std + tree_std)
print("and the two spreads "
      + ("overlap — treat this as a tie until you have more folds" if overlap
         else "do not overlap — the difference is larger than the noise"))

**Write the result down here, in your own words**, in two or three sentences. Name the winner, give
both numbers with their spreads, and say whether the difference is larger than the spreads.

Then the sentence that is actually being marked: **what would you tell a manager who asked for "a
deep learning solution" for this table?**

<div dir="rtl" align="right">

**اكتب النتيجة هنا بكلماتك** في جملتين أو ثلاث. سمِّ الفائز، واذكر الرقمين بمدى تفاوت كلٍّ منهما،
وقُل هل الفرق أكبر من مدى التفاوت.

ثم الجملة التي تُقيَّم فعلًا: **ماذا تقول لمدير طلب «حلًّا بالتعلّم العميق» لهذا الجدول؟**

</div>


**Your report:** _(winner, both numbers, and what you would say to the manager)_

<div dir="rtl" align="right">

**تقريرك:** _(الفائز، والرقمان، وما تقوله للمدير)_

</div>


## Section 3 — Stretch: batch norm, and the batch size that breaks it  (≈30 min)

Open-ended. Lower expectation of completeness.

`nn.BatchNorm1d` normalises each layer's inputs across the batch, so the next layer sees a stable
distribution instead of one that shifts every step. It buys faster training, more tolerance of a
badly chosen learning rate, and a mild regularising effect for free.

Add it, rerun, and record the change. Then set `batch_size=2` and rerun again.

At a batch of 2 you are normalising by the mean and variance of **two numbers**, which is not a
distribution — it is noise that lurches every step. Below about 8 batch norm stops helping and starts
hurting. Measure it rather than believing it, then write the one sentence explaining why, and answer
the closing question in the cell after.

<div dir="rtl" align="right">

## القسم الثالث — الإضافي: تطبيع الدفعات وحجم الدفعة الذي يكسره (نحو ٣٠ دقيقة)

مفتوح. والتوقّع في الإكمال أقل.

يُطبّع `nn.BatchNorm1d` مدخلات كل طبقة عبر الدفعة، فترى الطبقة التالية توزيعًا مستقرًا بدل توزيع
ينزاح في كل خطوة. ويشتري تدريبًا أسرع وتحمّلًا أكبر لمعدّل تعلّم سيّئ الاختيار، وأثرًا تنظيميًا خفيفًا
مجّانًا.

أضِفه وأعِد التشغيل وسجّل التغيّر. ثم اضبط `batch_size=2` وأعِد التشغيل مرة أخرى.

فعند دفعة من ٢ تكون تُطبّع بمتوسط وتباين **رقمين**، وهذا ليس توزيعًا بل ضوضاء تترنّح في كل خطوة. ودون
الثمانية تقريبًا يتوقّف تطبيع الدفعات عن النفع ويبدأ بالضرر. قِسه ولا تصدّقه، ثم اكتب الجملة الواحدة
التي تفسّر السبب، وأجب عن السؤال الختامي في الخليّة بعدها.

</div>


In [ ]:

# TODO: Three runs: no batch norm, batch norm, batch norm at batch_size=2.
# مهمة: ثلاثة تشغيلات: بلا تطبيع دفعات، ومع تطبيع، ومع تطبيع بحجم دفعة ٢.

print(batch_norm_table.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

**Your closing answers**, and the second one is the week's exit question:

1. Why does batch norm degrade at `batch_size=2`, and what would you use instead if your batches had
   to be that small?
2. **Given this week's numbers** — Monday's tree, Thursday's network, today's comparison — what kind
   of data would you reach for a neural network for, and what kind would you reach for a tree for?

<div dir="rtl" align="right">

**إجاباتك الختامية**، والثانية هي سؤال خروج الأسبوع:

١. لماذا يتدهور تطبيع الدفعات عند `batch_size=2`، وماذا كنت ستستخدم بدلًا منه لو اضطُرّت دفعاتك أن
   تكون بهذا الصغر؟
٢. **بالنظر إلى أرقام هذا الأسبوع** — شجرة الاثنين، وشبكة الخميس، ومقارنة اليوم — لأي نوع من البيانات
   تمدّ يدك إلى شبكة عصبية، ولأي نوع تمدّها إلى شجرة؟

</div>


## Save your artefacts

Two files, and the first one is the capstone's requirement in miniature.

- **`experiments.parquet`** — every run, every hyperparameter, both losses, the seed. **The capstone
  brief released today asks for exactly this table.** A report that says "we tuned the model and got
  0.84" is not defensible; one that ships this file is.
- **`best_model.pt`** — the state dict of your best configuration **and** the config that produced
  it, in one file. Weights without their config are not a model; you cannot rebuild the architecture
  to load them into.

Then reload it and confirm the reloaded copy reproduces the recorded score exactly. A checkpoint
nobody has reloaded is a checkpoint that does not work yet.

<div dir="rtl" align="right">

## احفظ مخرجاتك

ملفان، والأول هو متطلّب مشروع التخرّج مصغّرًا.

- **`experiments.parquet`** وفيه كل تشغيل وكل معامل والخسارتان والبذرة. **وكرّاسة مشروع التخرّج
  المنشورة اليوم تطلب هذا الجدول بالضبط.** فالتقرير الذي يقول «ضبطنا النموذج فحصلنا على ٠٫٨٤» لا
  يمكن الدفاع عنه، والذي يُرفق هذا الملف يمكن.
- **`best_model.pt`** وفيه `state_dict` أفضل إعداد **والإعداد** الذي أنتجه، في ملف واحد. فالأوزان بلا
  إعدادها ليست نموذجًا؛ إذ لا تستطيع إعادة بناء المعمارية لتحميلها فيها.

ثم أعِد تحميله وتأكّد أن النسخة المُعاد تحميلها تُعيد إنتاج النتيجة المسجّلة بالضبط. فنقطة الحفظ التي
لم يُعِد أحد تحميلها هي نقطة حفظ لا تعمل بعد.

</div>


In [ ]:
experiments_path = ARTEFACT_DIR / "experiments.parquet"
experiments.to_parquet(experiments_path, index=False)

# The best configuration, retrained with early stopping, saved with its config.
# Pulled back out of the DataFrame, so the numeric types are numpy's. torch wants
# Python ints, and this is the kind of edge that only appears when you reload a config.
best_config = {key: best_run[key] for key in BASE}
for key in ("width", "batch_size", "epochs", "seed"):
    best_config[key] = int(best_config[key])
best_config["dropout"] = float(best_config["dropout"])
best_config["lr"] = float(best_config["lr"])
best_config["batch_norm"] = bool(best_config["batch_norm"])

final_history, final_model, final_best_state = train(**best_config, patience=20)
final_model.load_state_dict(final_best_state)
recorded_loss, recorded_accuracy = score(final_model, X_val, y_val)

model_path = ARTEFACT_DIR / "best_model.pt"
torch.save({"state_dict": final_best_state,
            "config": best_config,
            "val_loss": recorded_loss,
            "val_accuracy": recorded_accuracy,
            "credit_default_auc": {"mean": network_mean, "std": network_std},
            "tree_baseline_auc": {"mean": tree_mean, "std": tree_std,
                                  "model": tree_baseline["best_model"]}},
           model_path)

# Reload from scratch — a checkpoint nobody reloaded is a checkpoint that does not work.
checkpoint = torch.load(model_path, weights_only=False)
rebuilt = build_model(checkpoint["config"]["width"], checkpoint["config"]["dropout"],
                      checkpoint["config"]["batch_norm"], checkpoint["config"]["seed"])
rebuilt.load_state_dict(checkpoint["state_dict"])
reloaded_loss, reloaded_accuracy = score(rebuilt, X_val, y_val)

reloaded_experiments = pd.read_parquet(experiments_path)
print(f"Saved {experiments_path}")
print(f"  {len(reloaded_experiments)} runs, {len(reloaded_experiments.columns)} columns")
print(f"\nSaved {model_path}")
print(f"  config: {checkpoint['config']}")
print(f"  recorded validation loss: {recorded_loss:.6f}")
print(f"  reloaded validation loss: {reloaded_loss:.6f}")
print(f"  reproduces exactly: {np.isclose(recorded_loss, reloaded_loss, atol=1e-9)}")

## Sanity check

Run this last. Every check that fails tells you what to fix and why.

The fourth one is the habit this lab exists to enforce: it walks every pair of rows in your
experiment log and fails if any two of them differ by more than one hyperparameter. The instruction
has never been enough on its own.

<div dir="rtl" align="right">

## فحص النتائج

شغّل هذه الخليّة أخيرًا. كل فحص يفشل يخبرك بما تُصلحه ولماذا.

والفحص الرابع هو العادة التي وُجد هذا المعمل لفرضها: فهو يمرّ على كل صف من سجلّ تجاربك مقابل خط الأساس ويفشل إن
حرّك أحدها مقبضين. فالتعليمة وحدها لم تكفِ يومًا.

</div>


In [ ]:
# --- Sanity checks ----------------------------------------------------------------

check(overfit_history.iloc[-1]["train_loss"] < 0.05
      and overfit_history.iloc[-1]["val_loss"] > overfit_history["val_loss"].min() * 1.2,
      f"task 2.1 must actually overfit: training loss below 0.05 (got "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}) and a final validation loss well "
      f"above the best one ({overfit_history.iloc[-1]['val_loss']:.4f} vs "
      f"{overfit_history['val_loss'].min():.4f})",
      f"يجب أن تُفرِط المهمة ٢٫١ في المطابقة فعلًا: خسارة تدريب دون ٠٫٠٥ (والناتج "
      f"{overfit_history.iloc[-1]['train_loss']:.4f}) وخسارة تحقّق نهائية أعلى بوضوح من "
      f"أفضلها ({overfit_history.iloc[-1]['val_loss']:.4f} مقابل "
      f"{overfit_history['val_loss'].min():.4f})")

check(train_worse,
      f"dropout must make the TRAINING loss worse — that is the regularisation doing its "
      f"job. Without dropout {comparison.iloc[0]['final_train']:.4f}, with dropout "
      f"{comparison.iloc[1]['final_train']:.4f}. (Whether it improved validation is the "
      f"experiment, and on this data it did not.)",
      f"يجب أن يُسيء Dropout خسارة **التدريب** — فهذا هو التنظيم يعمل عمله. فبلا Dropout "
      f"{comparison.iloc[0]['final_train']:.4f} ومعه {comparison.iloc[1]['final_train']:.4f}. "
      f"(أما هل حسّن التحقّق فتلك هي التجربة، ولم يحسّنه على هذه البيانات.)")

check(same_start,
      f"all four optimiser runs must start from identical weights, or you are comparing "
      f"initialisations. Their training losses before the first step were "
      f"{optimiser_table['loss_before_training'].round(6).tolist()}",
      f"يجب أن تبدأ تشغيلات المُحسِّنات الأربعة من أوزان متطابقة، وإلا فأنت تقارن تهيئات. "
      f"وخسائرها قبل الخطوة الأولى كانت "
      f"{optimiser_table['loss_before_training'].round(6).tolist()}")

hyperparameters = list(BASE)
baseline_row = experiments[experiments["changed"] == "-"].iloc[0]
offenders = []
for _, row in experiments.iterrows():
    differing = [k for k in hyperparameters if row[k] != baseline_row[k]]
    if len(differing) > 1:
        offenders.append((row["run"], differing))
check(len(experiments) >= 8 and not offenders,
      f"experiments.parquet needs at least 8 rows (has {len(experiments)}) and every row "
      f"must differ from the baseline by at most one hyperparameter. Offending runs: "
      f"{offenders[:3]}",
      f"يحتاج `experiments.parquet` ثمانية صفوف على الأقل (الموجود {len(experiments)}) "
      f"وأن يختلف كل صف عن خط الأساس بمعامل واحد على الأكثر. والتشغيلات المخالفة: "
      f"{offenders[:3]}")

check(restored_loss <= final_loss,
      f"early stopping must restore weights that score at least as well as the final "
      f"epoch's — restored {restored_loss:.4f} vs final {final_loss:.4f}. If the restored "
      f"weights are worse, you did not call load_state_dict(best_state).",
      f"يجب أن يستعيد الإيقاف المبكر أوزانًا لا تقلّ نتيجتها عن الحقبة الأخيرة — المستعادة "
      f"{restored_loss:.4f} مقابل النهائية {final_loss:.4f}. فإن كانت المستعادة أسوأ فأنت "
      f"لم تنادِ `load_state_dict(best_state)`.")

check(np.isclose(recorded_loss, reloaded_loss, atol=1e-9),
      f"best_model.pt must reload and reproduce its recorded score exactly — recorded "
      f"{recorded_loss:.8f}, reloaded {reloaded_loss:.8f}",
      f"يجب أن يُعاد تحميل `best_model.pt` وأن يُعيد إنتاج نتيجته المسجّلة بالضبط — المسجّلة "
      f"{recorded_loss:.8f} والمُعاد تحميلها {reloaded_loss:.8f}")

check(0.70 < network_mean < 0.85 and network_std > 0
      and "tree_baseline_auc" in checkpoint,
      f"the network-versus-tree comparison must be recorded with both spreads: network "
      f"{network_mean:.4f} +/- {network_std:.4f}, tree {tree_mean:.4f} +/- {tree_std:.4f}",
      f"يجب أن تُسجَّل مقارنة الشبكة بالشجرة بمدى تفاوت كلٍّ منهما: الشبكة "
      f"{network_mean:.4f} ± {network_std:.4f}، والشجرة {tree_mean:.4f} ± {tree_std:.4f}")

report()

## What's next

That is week 3. You arrived able to call `.fit()` and you leave having built the thing `.fit()` calls,
proved it correct against a numerical gradient, rebuilt it in PyTorch, and then measured it honestly
against a model with no neurons in it.

The last part is the one worth carrying. **On this table, the boosted tree won**, and your report says
so. That is not a failure of the week — it is the week's actual finding, and the ability to produce it
is worth more than another two points of AUC would have been.

Next week (**W4D1**) the data changes to the kind depth was built for: images. 150,000 pixels with
structure between them, no named columns, and nothing a tree can do with them at all. Convolutional
networks, and then fine-tuning a pretrained one — which is how every real vision system is built.

Three things travel with you:

- **The five-line loop**, unchanged, for every model in weeks 4 to 7.
- **`experiments.parquet`** — the shape of the log your capstone report needs. The brief is out today;
  read it tonight and start the log on day one, not the night before submission.
- **The diagnosis vocabulary.** Divergence, a flat curve at `ln k`, a widening gap. Those three
  readings apply unchanged to every model you will train from here.

<div dir="rtl" align="right">

## ماذا بعد

هذا هو الأسبوع الثالث. وصلت وأنت تعرف مناداة `.fit()` وتخرج وقد بنيت ما تناديه `.fit()`، وأثبتّ
صحّته مقابل اشتقاق عددي، وأعدت بناءه بـ PyTorch، ثم قِسته قياسًا صادقًا مقابل نموذج لا عصبون فيه.

والجزء الأخير هو الجدير بالحمل. **فعلى هذا الجدول فازت الشجرة المعزَّزة**، وتقريرك يقول ذلك. وهذا ليس
فشلًا للأسبوع، بل هو نتيجته الفعلية، والقدرة على إنتاجها أنفع مما كانت ستنفع نقطتان إضافيتان من
المساحة تحت المنحنى.

وفي الأسبوع القادم (**الأسبوع ٤ اليوم ١**) تتغيّر البيانات إلى النوع الذي بُني العمق لأجله: الصور.
مئة وخمسون ألف بكسل بينها بنية، وبلا أعمدة مُسمّاة، ولا شيء تستطيع الشجرة فعله بها إطلاقًا. الشبكات
الالتفافية، ثم الضبط الدقيق لشبكة مُدرَّبة مسبقًا — وهي طريقة بناء كل نظام رؤية حقيقي.

وثلاثة أمور تسافر معك:

- **حلقة الأسطر الخمسة** بلا تغيير، لكل نموذج في الأسابيع من الرابع إلى السابع.
- **`experiments.parquet`** وهو شكل السجلّ الذي يحتاجه تقرير مشروع تخرّجك. والكرّاسة صدرت اليوم،
  فاقرأها الليلة وابدأ السجلّ من اليوم الأول لا في ليلة التسليم.
- **مفردات التشخيص:** التباعد، والمنحنى المسطّح عند `ln k`، والفجوة المتّسعة. وهذه القراءات الثلاث
  تنطبق بلا تغيير على كل نموذج تدرّبه من هنا.

</div>
